# Part 3｜矩陣資料的維度與索引（以影像為例）
在 Part 1 中，我們認識了Python 中與影像有關的資料型別與基本結構，也學會使用 list 與迴圈讓程式可以有條例的運作。但是當資料變的複雜，例如影像、時間序列或者多維資料時，單純使用 list 已經不足以描述這些資料的結構。  
本節，我們回到 Python 中另一種核心的資料形式——**矩陣（NumPy array）**，並以影像為例，學習如何理解資料的維度，以及如何透過索引正確地操作資料。

---
### **3.1 矩陣的維度與影像的關連**  
當資料以 Numpy 陣列(ndarray) 的形式存在時，第一個要理解的，就是它的**維度數量與排列順序。**  
以下將以幾種常見的資料維度為例，說明矩陣在不同維度下的排列方式:
#### **常見的資料維度（Common Dimensions）**  
#### **1. 1D array**
* **概念：** 一條具有順序性的資料序列。
* **例子：** 常用來表示具有順序性的資料，例如光譜資料，或隨時間變化的訊號。

#### **2. 2D array (灰階影像)**
* **概念：** 資料沿著兩個維度排列而成的矩陣結構。
* **常見例子：** 平面資料，例如單層灰階影像
* **NumPy 表示法/維度順序：`(Y, X)`**
  * **Y (Row):** 第一個維度的大小(影像中對應高度)。
  * **X (Column):** 第二個維度的大小(影像中對應寬度)。

#### **3. 3D array**
* **概念：** 在 2D 資料的基礎上，多了第三個維度。這個第三個維度的意義，取決於資料的來源與用途。
* **常見例子一： Z-stack (多層影像)**  
    多張 2D 影像沿著第三個維度堆疊。  
    維度順序：`(Z, Y, X)`     
* **常見例子二： Multi-channel image(多通道影像)**  
    同一個空間位置，包含多個通道資料。  
    維度順序：可以是 `(C, Y, X)` 或 `(Y, X, C)`    
    >實際順序取決於檔案格式與讀取方式。

* **常見例子三：Color Image (RGB 彩色影像)**  
    單層影像，但每個像素位置包含 R、G、B 三個數值。  
    維度順序：`(Y, X, C)`
  
#### **4. 4D array**
* **概念：** 具有四個維度的資料陣列。  
* **常見例子：**  
   科學影像中最常見的結構，結合了「空間深度(Z)」與「通道(C)」的資訊。  
   最常見的維度順序：`(Z, Y, X, C)`
* **⚠️ 實務陷阱：**
    讀取 `.tif` 檔時，會受到檔案格式與讀取工具影響，讓實際排序可能為 `(Z, C, Y, X)` 或其他變形。  
  > **小撇步：** `.shape` 可以查詢矩陣維度。實作時，永遠以實際查詢結果為準，再根據數值的大小（例如 3 可能是 Channel，50 可能是 Z 軸）來判斷每個維度代表的意義。

#### **🔍 總結對照表(常見情況)**

| 影像類型 | 維度數 | 常見排列形式（例） | 備註 |
| :--- | :---: | :--- | :--- |
| 單層灰階 | 2D | `(Y, X)` | 最基本的矩陣形式 |
| 單色 Z-stack | 3D | `(Z, Y, X)` | 每層只有一個通道 |
| 單層 RGB | 3D | `(Y, X, C)` | C = 3，具有顏色語義|
| 多通道 Z-stack | 4D | `(Z, C, Y, X)`或 `(Z, Y, X, C)` | 實際順序取決於檔案與讀取方式 |

---

### **3.2 從影像檔案到 NumPy 陣列（ndarray)**   
從這裡開始，我們將實際讀取影像檔案，確認它在 Python 中所呈現的矩陣結構與維度，當作後續所有矩陣操作的起點。
在實務上，影像檔案讀取需要搭配專門的影像處理模組，而模組種類很多，各有適用的領域。   
本課程將以**NumPy + scikit-image + matplotlib + napari** 作為主要示範與操作的核心組合!!

---
#### **Step 0:準備工具 (Import Modules)**  
 我們透過以下程式碼，載入需要的模組與工具 (Import Modules)

In [2]:
#實務上我們有時候會以 as 來指定模組的縮寫，讓工具呼叫更容易
import numpy as np
from skimage.io import imread 
import matplotlib.pyplot as plt
import napari  

#### **Step 1:指定資料來源（檔案路徑）**  
定義檔案路徑時，請務必注意以下四個關鍵，否則 Python 會找不到你的圖片。  
* **路徑必為字串:**  
  頭尾需有引號包裹，例如：`'path/to/img.tif'`。
* **懶人必備 'r' 修正法:**  
  * 在引號前加上小寫 r (Raw string)，可以直接套用從 Windows 複製來的路徑，不必手動修改反斜線 \。
  * 範例：`img_path = r'C:\Users\Desktop\cell.png'`。  
* **優先使用「相對路徑」:**
  * `./` 代表「程式碼目前的位置」。
  * 好處：整份資料夾移動到哪都能跑，不用每次都改長長一串的地址。
  * 範例：`img_path = r'./data/my_image.tif'`。
* **一定要寫副檔名:**  
  Python需要完整的檔案地址，結尾必須包含 .tif、.png 或 .jpg 等。

為了方便呼叫，我們通常將路徑存入變數中!!


In [ ]:
#【任務 1】：使用相對路徑載入影像

# 請建立一個變數 filePath，並設定為正確的影像路徑
# 檔案位置：DemoImg 資料夾中的 Mito2

# 注意事項：
# 1. 路徑必須存到變數 filePath
# 2. Windows 系統請在字串前加上 r，避免 \ 被當成跳脫字元（例如 \n）
# 3. 請確認你輸入的檔名與實際檔案完全一致

filePath = _____________
print(filePath)

#### **Step 2:載入影像資料(imread)**
這裡我們使用 `skimage.io` 中的 `imread` 將影像資料載入 Python，並且將資料存入變數 `img`。

In [10]:
#【任務 2】：正式載入影像
# 1. 請填入 imread 並傳入變數 filePath，將影像存入變數 img
# 2. 使用 type () 函數觀察 img 的資料型別
img = _____ (_____)
print("資料型態為", type(img))
# 從這一刻開始，影像已經變成 Python 中的矩陣資料!!。

Type: <class 'numpy.ndarray'>


#### **Step 3:檢查資料維度與視覺化**
對於 NumPy 陣列來說，除了實際儲存矩陣中的數值之外，還會同時保留描述這個矩陣結構的資訊。其中，資料的維度會被記錄在陣列的 `shape` 屬性中。因此，我們可以用 `.shape` 這種方式來獲取資訊!!

In [ ]:
#【任務 3】：檢查維度順序 
# 提示：請使用 .屬性 的方式來獲取影像結構資訊
Dimension = img.__________

print("img 的維度為:", Dimension)

補充說明:
>除了維度(shape)外，Numpy 陣列還有其他的核心屬性:包括ndim（維度數）、dtype（資料型態）、size（總元素>數量）可以用 `.屬性`的方式獲取。

在確認維度結構後，我們可以用簡單的視覺化方式，直接檢查影像內容是否符合預期。
這裡我們可以利用 `matplotlib.pyplot` 中的 `plt.imshow()` 來顯示圖片:

In [ ]:
# 搭配 cmap 調整 LUT (Look-Up Table)
plt.imshow(img, cmap='gray')
plt.show()

補充說明:
> `cmap` 參數可以調整顯示時的顏色對應方式，有助於在檢查影像時觀察結構與弱訊號。

### **3.3 高維影像的檢視限制：為什麼需要 napari**  
當影像包含多個切片或多個通道時，資料會成 3D 或 4D 的 NumPy 陣列。在這種情況下， `matplotlib` 並不適合檢視資料，因此我們會改用更適合高維影像的視覺化工具—— napari。  

---
#### **3.3.1 Napari 的運作模式(viewer & layer)**  
Napari 的運作方式是透過 viewer負責顯示畫面，而實際資料則以 layer 的形式加入其中。  
常見的 layer 有以下幾種: 
| Layer | 代表資料 | 
|:---------| :-------:  |
|Image | 影像資料 |
| Labels| 分隔結果、整數標籤影像 |
| Points | 點座標(例如細胞位置) |
| Shapes | ROI / 線 | 
| Surface | 3D mesh |  
>不論是哪一種 layer，這些資料都可以再被轉成陣列(array) 形式進行後分析


#### **3.3.2 從 Python 推送高維資料進入 napari**  
在這個範例中，我們將要讀取一張包含多個維度的影像檔案，並將其推送入 napari檢視。


In [ ]:
# 【任務 4】：讀取 4D 原始影像並觀察其維度
# 請使用 DemoImg 資料夾下的 4D_DataSet.tif
# 1. 請填入影像路徑，並使用 imread 讀取影像
filePath = r"______"
img = imread(filePath)

# 2. 使用 shape 屬性觀察影像結構
print("shape:", img._____)

接著，建立一個 viewer，並加入影像資料時，透過 `channel_axis` 參數明確指定哪一個維度代表通道（channel），以避免 napari 將影像誤判為 RGB。

In [ ]:
#請執行以下程式碼
viewer = napari.Viewer()
viewer.add_image(img, channel_axis=-1, name="Split_4D_DataSet")

Napari 會根據陣列的維度結構，自動建立對應的影像 layer，讓我們能進行互動式檢查。
補充說明:
>若要加入其他類型的 layer，可以使用 `.add_labels()`、`.add_points()`、`.add_shapes()`...等方法。

#### **3.3.3 從 napari 抓取資料**  
接下來，我們要反過，將 napari 中的資料取回，讓我們能進行後續影像分析。  
首先，從 `viewer.layers` 中取得想要的 layer（可以使用名稱或 index 指定）：

In [22]:
#請執行以下程式碼
imglayer = viewer.layers["Split_4D_DataSet"]
# 或
# img_layer = viewer.layers[0] 

然後，透過 `.data` 取出對應的 NumPy陣列:

In [29]:
#請執行以下程式碼
imgData = imglayer.data
print(imgData.shape)

(13, 1000, 1000)


此時，imgData 即為一個 NumPy array，可以搭配 NumPy、scikit-image 或 matplotlib 進行後續處理。
>**補充說明**
>1.  由於多通道影像在載入 napari 時已被拆分為多個 Image layer，此處取得的 imgData 僅對應單一通道的影像資料。
>2. `viewer.layers[index]` 中的 `index` 代表 layer 在右側清單中的順序，其值可能會隨新增或拖曳而改變。 
>3. `layer.data` 回傳的是陣列，若直接修改內容，將會影響原始資料。  


---

### **3.4 矩陣切片 (Slicing): 從提取開始**
在矩陣操作中，切片(Slicing) 是我們進行資料提取的起點。面對如影像資料的高維結構，例如 `(Z, Y, X, C) `，我們可以透過切片，選擇並重組需要的資料範圍，讓後續的影像分析流程更有效率。

---
#### **3.4.1 切片的核心語法** 
我們在Part 0 學過用**索引值**從 List 中提取資料，NumPy 矩陣的切片方式延續這個概念。不同的是，在多維陣列中，一定要分別指定每一個維度的索引方式。
以影像資料常見的 `(Z, Y, X, C)  `結構為例，其基本切片語法如下：
```Python 
  img[z, y, x, c]  
```
括號中`[ ] `，每一個位置對應一個維度，每一個維度有三種索引方式：
* **單一索引值**  
  選取該維度中的某一層或某一通道，該維度會在結果中被移除(維度減少)。
  
* **冒號 `: `**  
   代表選取該維度的全部資料，該維度會被完整保留。
   
* **區間`start:end`**  
    選取部該維度中部分範圍(從 start 到 end -1 )。
    
    
>**補充說明:**
>不論是哪一個維度，所有索引值皆從 0 開始。



####  **3.4.2 降維任務: 從 4D 中提取不同維度的數據**
在進行操作前，我們要先確認影像資料在 NumPy 中的維度結構。

In [24]:
print("影像資料 shape :", img.shape)

影像資料 shape : (13, 1000, 1000, 3)


**補充說明:** 示範影像的維度順序為 (Z, Y, X, C)。 &nbsp;實際操作時，請以 `img.shape` 為準!!!

---

**A. 取單一通道的 Z-stack image (保留Z軸)**  
 如果我們只想看某一個顏色（例如 channel 1 的 DAPI 訊號），我們可以這樣寫:

In [ ]:
#【任務 5】：完成 slicing，取得第一個 channel（保留 Z 軸）
img_3d = img[:, :, :, ____]

print("提取後 3D 影像的資料：", img_3d.shape)

 **B. 提取 2D 平面 (指定單一層數與單一通道)**  

In [ ]:
# 【任務 6】：完成 slicing，取得第 7 層 + 第 3 個 channel

img_2d = img[____, :, :, ____]
print("提取後 2D 影像的身材：", img_2d.shape)

**C. 區間切片（練習 start:end）**

In [ ]:
# 【任務 7】：完成 slicing，取第 3 層到第 7 層（共 5 層）+ 第二個 channel
img_part = img[____:____, :, :, ____]

print("提取後影像的資料：", img_part.shape)

**⚠️ 常見的錯誤 - IndexError**  
>NumPy 的索引是 從 0 開始，不是從 1。因此在 slicing 或 indexing 時，索引值必須落在合法範圍內。  

錯誤範例：  
如果數據只有 3 個通道（Size = 3），嘗試選取 `img[1, :, :,4] `會導致報錯，因為通道索引只能是 0 或 1。

### **3.5 矩陣裁剪(Crop): 以子矩陣定義感興趣區域（ROI）**  
在前一節裡，我們介紹了 NumPy 矩陣切片時的三種索引方式。本節開始，我們將實際使用**第三種索引方式──區間索引（`start : end`）**
，來從影像中選取一塊連續的區域，作為後續分析的對象。  
在影像分析中，這樣的操作通常被稱為「裁剪（crop）」。從矩陣的角度來看，裁剪的結果就是從**原始影像中取出一個子矩陣**，用來定義我們感興趣的區域（ROI, Region of Interest）。這樣做的好處是我們可以縮小範圍，排除無效的背景、節省電腦運算記憶體，並讓定量分析更準確。

---
當我們用**區間索引** 來裁剪影像時，需要特別注意以下兩個座標規則。:
#### **裁剪的兩大鋼鐵規則** 
1. **原點(0,0) 在左上角:**
* 影像的第一個像素從 **左上角** 開始計算。
* 座標順序為 `[Y, X]`，即: 先決定「上下位置」，再決定「左右位置」。
2. **不包含結尾索引(Exclusive End):**
* 語法`[start : end]` 會包含 `start` 位置，但不包含 `end`位置。
* 公式: `裁剪後尺寸 = end − start`

---
接下來，我們以不同維度影像為例，實際使用「區間索引（start:end）」來裁剪影像中的特定區域，並透過 napari viewer 來檢視裁剪的結果。

#### **A. 2D 影像裁剪**  
目標： 從 img_2d 中裁切出 Y 軸 $50 \sim 499$ 像素、X 軸 $460 \sim 899$ 像素的區域。

In [ ]:
# 【任務 8】：手動裁切 2D 影像
# --- 請在下方填空完成裁切 ---
roi_2d = img_2d[50:500, 460:900] 

# 檢查身材：預期應為 (450, 440)
print("裁切後 ROI 的身材：", roi.shape)

# --- 檢視結果 (直接執行即可) ---
try:
    viewer.close() # 如果已經有開啟的視窗，就先關掉它
except:
    pass # 如果沒視窗，就跳過直接往後執行
    
viewer = napari.Viewer()
viewer.add_image(img_2d, name="original")
viewer.add_image(roi, name="roi")

#### **B. 3D 影像裁剪**  
這一節，我們來試試看如何同時在 Z、Y、X 三個維度上使用區間索引，裁剪出一個立體的 ROI。

現在，我們手上已經有一個 3D 影像矩陣 `img_3d`，它由 13 層 組成，每層大小為 $1000 \times 1000$。在進行裁切填空時，請依照 (Z 層數, Y 高度, X 寬度) 的順序來思考。

**目標裁出：**  
**(1) Z 軸:** 從第 2 層到第 10 層(共 9 層)。   
**(2) x 軸:** 取像素座標範圍 518 ~ 821。   
**(3) Y 軸:** 取像素座標範圍 174 ~ 439。    
 

In [ ]:
# 【任務 9】：手動裁切 3D 影像
# --- 請在下方填空完成 3D 裁切 ---
roi_3d = img_3d[1:10, 174:440, 518:822]

# 檢查身材：預期應為 (9, 266, 304)
print("裁切後 3D ROI 的身材：", roi_3d.shape)

# --- 檢視結果 (直接執行即可) ---
try:
    viewer.close() # 如果已經有開啟的視窗，就先關掉它
except:
    pass # 如果沒視窗，就跳過直接往後執行
    
viewer = napari.Viewer()
viewer.add_image(img_3d, name="original_3d")
viewer.add_image(roi_3d, name="roi_3d")

>**補充說明：**
>在實務上，處理多通道的高維度影像時，通常會先選擇一個分析通道，再進行後續操作；相關概念將在 Mask 與 Boolean 運算章節中說明。


### **3.6 影像遮罩(Masking) 與 布林運算**  
Crop 是根據指定的影像範圍(座標位置)來取出一個方形ROI。不過，在實際的影像分析中，我們更常用「像素的數值是否符合條件」，來決定哪些像素要被保留下來、哪些應該被排除。  
這時候，就是要到 **影像遮罩（mask)** 。 與ImageJ 不同的是，在 Python 中，mask 是一張由 True 與 False 組成、用來標記感興趣區域的影像。



#### **3.6.1 Mask 的核心語法 : 比較運算**  
Mask 最常見的建立方式，是對影像做逐像素比較，例如:
```Python
 mask = img_2d > Value
```
 
>這行語法代表: (1)對影像中 **每個像素** 做比較 &nbsp; (2)符合條件的像素標記為 `True` &nbsp;(3)不符合的標記為 `False`

In [ ]:
#【任務 10】：動手做遮罩
#1.請建立一個遮罩 mask，找出 roi_2d 中數值高於 4800 的部分。
#2.執行後，我們印出遮罩裡對應原始影像較亮的區域（Y 索引 170~172, X 索引 190~220）：


**關鍵概念:** 建立 mask 不會改變影像的空間範圍（shape 不變）

---
#### **3.6.2 Mask 的使用方式：影像遮罩 vs 資料篩選**  
Mask 有兩種常見但不同用途的用法
- **影像遮罩（Image Masking）：** 保留符合條件的像素，其他設為 0（影像大小不變，內容更乾淨）
- **資料篩選（Data Selection）：** 只取出符合條件的像素值（不再是影像）。


接下來，我們以 2D 影像的 Mask 為例，來練習不同的使用方式:

**方法一：影像遮罩（輸出仍是 2D 影像）**  
利用 mask，將「不符合條件」的像素設為 0。    
   **寫法 A：使用反向遮罩 `~mask`**  
   ```Python  
   filtered = img_2d.copy()   # 先建立影像副本，避免改到原始影像
   filtered[~mask] = 0        # 不符合條件的位置設為 0
   ```
   > `~mask` 代表將 True / False 反轉
  
   **寫法 B：使用 `np.where()`**    
   ```Python  
   filtered = np.where(mask, img_2d, 0)
   ```
   > 若 mask 內的 pixel 為 True → 保留原始影像的像素值  
   > 若 pixel 為 False → 設為 0

   這兩種寫法效果相同：  
   影像大小(Shape)不變，，只是部分像素被設為 0!! 

In [ ]:
#請執行以下程式碼，觀察影像的維度變化。  
#寫法A:
filteredA =  roi_2d.copy()
filteredA[~mask] = 0
#寫法B: 
filteredB = np.where(mask,  roi_2d, 0)

print("原始影像 :",  roi_2d.shape)
print("mask :", mask.shape)
print("filteredA 與 filteredB :", filteredA.shape,  filteredB.shape)

# --- 檢視結果 (請執行以下程式碼，進入 napari 觀察) ---
try:
    viewer.close() # 自動清理舊視窗
except:
    pass
    
viewer = napari.Viewer()
viewer.add_image( roi_2d, name="0.原始影像", colormap="gray")
viewer.add_image(filteredA, name="1.遮罩處理(寫法A)", colormap="green")
viewer.add_image(filteredB, name="2.遮罩處理(寫法B)", colormap="red")

> **補充說明: 反向遮罩（`~mask`）的真正意義**  
>* `~` 在 NumPy 中是「逐元素的布林反轉運算」。  
>* `~mask`代表: 將 mask 中每一個像素位置的判斷結果做反轉（True ↔ False），而不是在操作「一整塊區域」。  
>* 在切片或索引操作中，**布林陣列是用來指定哪些位置會被選取**，只有判斷結果為 True 的位置才會被被保留或被修改。

**方法二 : 資料篩選（輸出是一維資料，不再是影像）**  
遮罩(mask) 也能當作資料篩選器，把所有判斷為 `True` 的像素值「撈出來」:  

In [ ]:
#請執行以下程式碼，觀察維度變化。
roi_values = roi_2d[mask]
print(roi_values.shape) 
#NumPy 回傳的是 一個 1D array，原本的 2D 空間結構 完全消失!! 

> **補充: 為什麼 `img_2d[mask]` 會降維**？  
> 因為在 NumPy 中，這種寫法代表「從陣列中取出所有符合條件的元素」，回傳的是一個一維的資料集合，而不是一張影像。

### **3.7 進階技巧：遮罩生產器的更多選項 (Comparison Operators)**
製作遮罩時，不只有 「大於」，根據不同的分析目的，我們可以靈活變換條件：    

| 運算子 |	意義 | 生物影像實務應用案例 |  
|:-------: |-----|:---------------| 
|>	| 大於 | 找出訊號高於背景的區域。 |  
|<	| 小於 | 找出影像中的暗部（例如明視野中的細胞）。 |  
|== | 等於 | 出標記過的特定細胞編號（如：只選出第 5 號細胞）。 |  
|!= | 不等於 | 排除特定的雜質或已知的錯誤標記。 |  

#### **進階組合技: 多重條件(Logical Combination)**  
當單一門檻無法滿足需求時，我們可以用 **括號** 將多個條件 `( 夾 )`起來 :  
* **區間選取(AND):** 
```Python
mask = (img > 200) & (img < 800)
#應用：只保留亮度適中的細胞，排除太暗的背景與太亮的雜訊。
```
* **聯集選取(OR):** 
```Python
mask = (img < 50) | (img > 900)  
#應用：同時抓出最暗與最亮的極端值區域。
```
>⚠️ 注意：在做複合判斷時，小括號 () 絕對不能省，否則 Python 會因為運算順序而報錯！

### **3.8 實作任務：DAPI 訊號的分析 Pipeline** 

我們將以一組 4D 影像資料 `4D_DataSet_reorder` 為例，整合前面所學的矩陣操作與 mask 概念，完成一個基礎的影像分析流程。

在這個任務中，我們會從 DAPI 通道出發，建立遮罩（mask），並進一步量測其面積與強度，最後透過 napari 視覺化分析結果。


**Step0: 載入套件與讀取影像資料**  
在開始分析之前，我們先載入需要的套件，並讀取本次使用的 4D 影像資料。

In [ ]:
import numpy as np
from skimage.io import imread
from skimage.filters import threshold_otsu
import napari

rawImg = imread(r"________________________")
print(rawImg.shape) #確認維度順序 # (____, ____, ____, ____)

**Step1.提取 DAPI 通道（降維）**   
在這個資料中，影像的維度順序為 (C, Z, Y, X)，而 DAPI 是第一個通道。  
請依照維度順序完成 slicing，將 DAPI 通道從 4D 影像中取出，並觀察輸出的 shape 變化。

In [ ]:
#請填入正確的索引資訊，並觀察輸出的維度變化。
dapi = rawImg[____,____,____,____]
print(dapi.shape) # 預期輸出 shape: (13, 1000, 1000)

**Step 2：計算 threshold 並建立 mask**  
我們先利用 skimage 自動計算 DAPI 影像的 threshold，再根據這個 threshold 建立遮罩（mask）

In [ ]:
th =  threshold_otsu(dapi)  
print(f"自動計算出的門檻值為: {th}")

# 請完成下列程式，建立「亮度大於 threshold」的 mask
nucleus_mask = ______________________

#檢查結果
print(nucleus_mask.shape)

**Step 3：利用 mask 篩選 DAPI 訊號**  
這一步的目的，是把 DAPI 影像中符合 mask 條件的像素值取出，作為後續量測面積與平均強度的基礎資料。

In [ ]:
# 請利用 mask 取出 DAPI 影像中符合條件的像素值
roi_values = dapi[________]

print(roi_values.shape)
print(roi_values[:5])   # 觀察前幾個值

**Step 4：量測面積與平均強度**  
完成 mask 建立後，我們可以進一步量測：  
- **面積（area）：** 符合條件的像素數量  
- **平均強度（mean intensity）：** 符合條件像素的平均亮度
> 這些量測可以直接用 NumPy 的 .sum() 和 .mean() 輕鬆完成。

In [ ]:
#請完成以下的程式碼
area = ________.sum()
mean_intensity = ________.mean()

print("Area:", area)
print("Mean intensity:", mean_intensity)

**Step 5：套用遮罩並檢視結果**   
最後，我們將 mask 套用回原始影像，並使用 napari 檢視處理結果。

In [ ]:
masked_img = dapi.copy() # 建立影像副本，避免修改原始資料

#將不符合條件的區域設為 0（只保留 mask 為 True 的像素）
masked_img[____________] = 0

#或者也可以使用 np.where() 達成相同效果
# masked_img = np.where(nucleus_mask, dapi, 0)

# --- 檢視結果 (請執行以下程式碼，進入 napari 觀察) ---
try:
    viewer.close() # 自動清理舊視窗
except:
    pass
    
viewer = napari.Viewer()
viewer.add_image( dapi, name="0.原始影像", colormap="gray")
viewer.add_image(masked_im, name="filtered image", colormap="green")


**透過這種方式，我們成功將背景噪音去除，僅保留符合統計門檻（Otsu）的訊號區塊。**

---
### **3.9 進階補充：連續索引與安全性 (Advanced Indexing)**    
在 Python/NumPy 中，妳可能會看到有人把提取與修改寫在同一行，這種寫法被稱為「連續索引 (Chained Indexing)」。

```Python
# 簡潔但具風險的寫法:
# 一次到位：提取第 0 頻道並同時套用遮罩歸零!!
MaskedDapi[:, 0, :, :][~nucleus_mask] = 0

```
**為什麼不建議這樣寫?**
1. 可讀性差 : 當維度增加（例如 4D 或 5D）時，一長串的中括號 `[]` 很容易數錯座標或維度。  
2. 副本 vs. 視圖 (Copy vs. View) 的陷阱：
   * NumPy 的運算機制有時會在切片步驟產生了「副本 (Copy)」，導致最後的賦值只改到了一個臨時變數，而原本的大圖完全沒變。
   * 這種錯誤通常不會跳出報錯訊息（Error Message），妳只會發現結果圖怎麼畫出來都跟原圖一樣，非常難 Debug
3. API 相容性：如同 ImageJ 等舊型影像處理 API，許多系統並不支援這種連續修改，必須透過「明確賦值」來確保數據寫回記憶體。
